# Selectividad Exams Parser

This script reads the content of the PDF files, to extract the information from files

## Libraries import

### Required packages for OCR processing:
**Ubuntu based:**
- tesseract-ocr
- tesseract-ocr-spa
- poppler-utils

**Archlinux based:**
- tesseract
- tesseract-data-spa

In [1]:
#!pip install pdf2image
#!pip install pytesseract

In [2]:
from datetime import datetime
from multiprocessing import Pool
from pathlib import Path
import os
import shutil

import pandas as pd
from pdf2image import convert_from_path
import pytesseract

In [3]:
# set root path
root = "/Users/daniel/code/clasificador_selectividad"
os.chdir(root)

## Functions

### *pdf_to_text*

Extract the text from pdf files, page by page

In [4]:
def pdf_to_text(file: Path) -> list:
    """
    Processes a single PDF file to extract its content as text.
    """
    images = convert_from_path(file)[1:]
    # parse each page
    pages = []
    for i, page in enumerate(images):
        pages.append(pytesseract.image_to_string(page, lang='spa'))
    return pages

### *extract_statement*
Extract the statement from a page in text format

In [5]:
def extract_statement(page: str) -> str:
    # split the content into lines
    text = page.split('\n')

    # list to store the processed lines
    extracted_text = []
    # start from the 4th line
    for line in text[4:]:

        # check the first word of the line
        first_word = line.replace('.', '').split(' ')[0].strip().upper()

        # stop at examen details line
        if first_word in ['SOCIALES', 'FISICA', 'MATES', 'MATEMÁTICAS', 'QUÍMICA', 'QUIMICA', 'QUIÍMICA']:
            extracted_text.append(line.strip())
            break
        
        # add the line to final_text
        else:
            if not line == '':
                extracted_text.append(line.strip())

    # join the list into a single string
    extracted_text = '\n'.join(extracted_text)

    return extracted_text

### *extract_exam_details*
Extract the exam details, like subject, year, exam and exercise number.

In [6]:
def extract_exam_details(statement : str) -> dict :
    # get last line with exam details
    lines = [i for i in statement.split('\n') if i != '']
    exam_details = lines[-1].lower()

    # strip whitespaces
    exam_details = (exam_details.replace('.', '')            
                    .replace(',', '')
                    .replace('(', '')
                    .replace(')', '').strip()
    )
    
    # split the words by ' '
    exam_details = exam_details.split(' ')

    # MATES CCSS
    # SOCIALES II. 2017 JUNIO. EJERCICIO 2. OPCIÓN A
    # 
    if exam_details[0] == "sociales":
        exam_details[0] = 'MATES_CCSS'
        # remove the second element in the list
        del exam_details[1]
    elif exam_details[0] == "matemáticas":
        exam_details[0] = 'MATES_II'
        # remove the second element in the list
        del exam_details[1]
    
       
    # exam dictionary
    details = ['subject', 'year', 'exam', 'exercise']
    exam_dict: dict[str, str | int | None] = dict.fromkeys(details)
    exam_dict = {k:None for k in details}

    ## fill dictionary with values
    
    # subject
    exam_dict['subject'] = exam_details[0]
    
    # year
    # SOCIALES II. PONENCIA 2009. EJERCICIO 2
    if exam_details[1] == 'ponencia':
        # switch ponencia and year
        exam_details[1], exam_details[2] = exam_details[2], exam_details[1]

    exam_dict['year'] = int(exam_details[1])

    # parse the exam string
    if exam_details[2].startswith('reserva'):
        exam_dict['exam'] = ' '.join(exam_details[2:4]).title()
        exam_index_start = 4
    else:
        exam_dict['exam'] = exam_details[2].title()
        exam_index_start = 3
    
    exam_dict['exercise'] = ' '.join(exam_details[exam_index_start:]).title()

    
    return exam_dict

### *generate_error_log*
generate a error log and add it to errors_timestamp.log file

In [7]:
def generate_error_log(file, page, error_type, statement = None, details = None):
    log = f'Error processing {error_type}, in {file.name} at page {page+2}\n'
    
    # add timestamp to log file
    current_datetime = datetime.now()
    formatted_dt = current_datetime.strftime("%d-%m-%Y")
    # crear error_logs folder
    if not Path("./error_logs").exists():
        Path("./error_logs").mkdir()
    with open(f'./error_logs/errors_{formatted_dt}.log', 'a') as f:
        f.write(log)
        f.write('*' * 10 + '\n')
        if statement:
            f.write(f'{statement}\n')
        elif details:
            f.write(f'{details}\n')
        f.write('*' * 10 + '\n')
    
    # print log to console without the new line
    print(log[:-2])

### *process_file*
Process a pdf file, extracting its information: statement and exam information 

In [8]:
def process_file(file: Path) -> pd.DataFrame:
    
    topic = file.stem.split(' - ')[-1]   
    
    # create a dict to store parsed values
    exercises_dict = dict.fromkeys(['subject', 'year', 'exam', 'exercise', 'statement'])
    # initialize the dict with empty lists
    exercises_dict = {k:[] for k in exercises_dict.keys()}

    pages = pdf_to_text(file)

    # parse each page in the pdf file

    print(f'Success parsing file: {file.stem}')
    
    for index, page in enumerate(pages):

        # extract statement from page
        try:
            statement = extract_statement(page)
        # error with extracting statement
        except:
            for line in statement.split('\n'):
                if line.startswith('www.emestrada.org'):
                    continue
            generate_error_log(file = file, page = index, error_type= 'statement', statement=page)
            continue

        # extract exam details
        try:
            details = extract_exam_details(statement)

            # store the details in the exercises dictionary
            for k,v in details.items():
                exercises_dict[k].append(v)
            
            # finally, add the statement text 
            exercises_dict['statement'].append(statement)
        
        # error with extracting details
        except:
            # pages ending with www.emestrada.org are pages with solution to exercise
            if not statement.endswith('www.emestrada.org'):
                generate_error_log(file = file, page = index, error_type= 'details', details = statement)
                output = f'File {file.stem} parsed but with errors'
                print(output)
            continue
        
    df = pd.DataFrame(exercises_dict)
    df['topic'] = topic
    df = df[["subject", "topic", "year", "exam", "exercise", "statement"]]

    return df

In [9]:
#process_file(Path("./pdf_files/quimica/enlace_quimico/2011 - Enlace Químico.pdf"))

### *process_folder*
Processes all files within a folder

In [28]:
def process_folder(folder_path:Path) -> pd.DataFrame:
    df = pd.DataFrame()
    
    
    # get the list of pdf files within the folder
    files = sorted( [i for i in folder_path.iterdir()
                        if i.suffix == '.pdf'] )
    
    # process each pdf file to extract its information
    for file in files:
        try:
            exercises_df = process_file(file)
            # each file contains a dataframe with columns:
            # subject, year, exam, exercise, topic
            df = pd.concat([df, exercises_df], axis = 0)

        except Exception as e :
            print(e)
        
    # create csv folder to store csv files
    ## concatenate folder path provided with a subfolder called csv_files
    csv_folder = Path(f"{root}/CSV_parsed")
    if not csv_folder.exists():
        csv_folder.mkdir()
        print(f"Folder '{csv_folder}' created.")
    else:
        print(f"Folder '{csv_folder}' already exists.")

    # create subfolder for subject
    subject = folder_path.parent.stem.replace(" - Temas", "")
    if not (csv_folder / Path(subject)).exists():
        (csv_folder / Path(subject)).mkdir()
        print(f"Subfolder '{subject}' created.")
    
    # export the output of the processed pdf files to a csv file
    output_file = csv_folder / Path(f'{subject}/exercises_{folder_path.stem}.csv')
    print(f"Saved file to: {output_file}")

    df.to_csv(output_file, index=False)
    
    return df

# Procesado de los archivos

## Inspeccionar la estructura de carpetas
Obtener objetos de tipo `Path` con las carpetas de cada tema dentro de cada asignatura.

In [16]:
# creación de diccionario con los temas disponibles
subject_folders = {}
subjects = [subject for subject in Path.iterdir(Path(root) / "pdf_files/") if subject.is_dir()]
for s in subjects:
    subject_folders[s.name.replace(" - Temas", "")] = [f for f in Path.iterdir(Path(root) / "pdf_files/" / f"{s.name}") if f.is_dir() ]

In [24]:
for k, v in subject_folders.items():
    print(f"Asignatura: {k}")
    for i in v[:3]:
        print(i)

Asignatura: Física
/Users/daniel/code/clasificador_selectividad/pdf_files/Física - Temas/campo_eléctrico_magnético
/Users/daniel/code/clasificador_selectividad/pdf_files/Física - Temas/ondulatorio
/Users/daniel/code/clasificador_selectividad/pdf_files/Física - Temas/física_cuántica_nuclear
Asignatura: Química
/Users/daniel/code/clasificador_selectividad/pdf_files/Química - Temas/ácido_base
/Users/daniel/code/clasificador_selectividad/pdf_files/Química - Temas/redox
/Users/daniel/code/clasificador_selectividad/pdf_files/Química - Temas/enlace_químico
Asignatura: Mates CCSS
/Users/daniel/code/clasificador_selectividad/pdf_files/Mates CCSS - Temas/probabilidad
/Users/daniel/code/clasificador_selectividad/pdf_files/Mates CCSS - Temas/programación_lineal
/Users/daniel/code/clasificador_selectividad/pdf_files/Mates CCSS - Temas/matrices_determinantes
Asignatura: Mates II
/Users/daniel/code/clasificador_selectividad/pdf_files/Mates II - Temas/sistemas_de_ecuaciones_lineales
/User

Es necesario normalizar por un tema específico de Mac OS

In [22]:
import unicodedata

def norm(s):
    return unicodedata.normalize("NFC", s)

subject_folders = {norm(k): v for k, v in subject_folders.items()}

In [30]:
for topic in subject_folders["Mates CCSS"]:
    process_folder(topic)

Success parsing file: 2001 - Probabilidad
Error processing details, in 2001 - Probabilidad.pdf at page 
File 2001 - Probabilidad parsed but with errors
Error processing details, in 2001 - Probabilidad.pdf at page 
File 2001 - Probabilidad parsed but with errors
Error processing details, in 2001 - Probabilidad.pdf at page 
File 2001 - Probabilidad parsed but with errors
Error processing details, in 2001 - Probabilidad.pdf at page 
File 2001 - Probabilidad parsed but with errors
Success parsing file: 2002 - Probabilidad
Error processing details, in 2002 - Probabilidad.pdf at page 
File 2002 - Probabilidad parsed but with errors
Error processing details, in 2002 - Probabilidad.pdf at page 1
File 2002 - Probabilidad parsed but with errors
Error processing details, in 2002 - Probabilidad.pdf at page 1
File 2002 - Probabilidad parsed but with errors
Success parsing file: 2003 - Probabilidad
Error processing details, in 2003 - Probabilidad.pdf at page 
File 2003 - Probabilidad parsed but with

In [51]:
lines = []
with open("CSV_parsed/errores_parseo_ccss.txt", encoding = "utf-8", mode = "r") as file:
    for line in file.readlines():
        if line.startswith("File"):
            if line not in lines:
                line = line.replace(" parsed but with errors\n", "")
                line = line.replace("File ", "")
                elements = line.split(" - ")
                lines.append(elements[1] + " - " + elements[0] + "\n")
lines = sorted(list(set(lines)))

with open("CSV_parsed/errores_parseo_ccss2.txt", encoding = "utf-8", mode = "w") as file:
    file.writelines(lines)

## Process pdf files

## Gather the csv files and combine them

### Get the path of all of the csv files:

In [55]:
csv_subject_folders = [i for i in Path(f"{root}/csv_parsed/").iterdir() if i.is_dir() ]
csv_file_paths = {}
for subject_path in csv_subject_folders:
    csv_file_paths[subject_path.stem] = list(subject_path.rglob("*.csv"))

In [56]:
csv_file_paths

{'Mates CCSS': [PosixPath('/Users/daniel/code/clasificador_selectividad/csv_parsed/Mates CCSS/exercises_sistemas_ecuaciones_lineales_ccss.csv'),
  PosixPath('/Users/daniel/code/clasificador_selectividad/csv_parsed/Mates CCSS/exercises_funciones_ccss.csv'),
  PosixPath('/Users/daniel/code/clasificador_selectividad/csv_parsed/Mates CCSS/exercises_probabilidad.csv'),
  PosixPath('/Users/daniel/code/clasificador_selectividad/csv_parsed/Mates CCSS/exercises_matrices_determinantes.csv'),
  PosixPath('/Users/daniel/code/clasificador_selectividad/csv_parsed/Mates CCSS/exercises_programación_lineal.csv'),
  PosixPath('/Users/daniel/code/clasificador_selectividad/csv_parsed/Mates CCSS/exercises_inferencia_estadística.csv'),
  PosixPath('/Users/daniel/code/clasificador_selectividad/csv_parsed/Mates CCSS/exercises_contraste_hipótesis.csv'),
  PosixPath('/Users/daniel/code/clasificador_selectividad/csv_parsed/Mates CCSS/exercises_distribución_binomial.csv')],
 'Física': [PosixPath('/Users/dani

### Lectura de los archivos csv y exportación a un único csv

In [57]:
for k, v in csv_file_paths.items():
    df = pd.DataFrame()
    for file in v:
        csv_df = pd.read_csv(file)
        df = pd.concat([df, csv_df])
    df.to_csv(f"{root}/CSV_parsed/{k}/ejercicios_{k}.csv", index = False)

## Revisión del archivo csv combinado de la asignatura

In [59]:
df = pd.read_csv(f"{root}/CSV_parsed/Física/ejercicios_Física.csv")

In [60]:
df.head()

,subject,topic,year,exam,exercise,statement
0,fisica,Campo eléctrico y magnético,2016,Junio,Ejercicio 1 Opción A,a) Campo eléctrico creado por una carga puntua...
1,fisica,Campo eléctrico y magnético,2016,Junio,Ejercicio 3 Opción B,"Una espira circular de 2,5 cm de radio, que de..."
2,fisica,Campo eléctrico y magnético,2016,Reserva 2,Ejercicio 1 Opción B,a) Fuerza magnética sobre una carga en movimie...
3,fisica,Campo eléctrico y magnético,2016,Reserva 2,Ejercicio 3 Opción A,"Un péndulo consta de una esfera de 20 g, carga..."
4,fisica,Campo eléctrico y magnético,2016,Reserva 3,Ejercicio 1 Opción A,a) Enuncie la ley de inducción electromagnétic...


In [61]:
df.topic.unique()

<StringArray>
['Campo eléctrico y magnético',                   'FIS-T3-2021',
                   'FIS-T3-2022',                   'FIS-T3-2023',
                   'FIS-T3-2024',                   'FIS-T3-2025',
                         'Ondas',   'Física cuántica y nuclear',
            'Campo gravitatorio',           'Óptica geométrica']
Length: 10, dtype: str